# Agentic Pipeline Observability & Cross-Indiscretion Matrix
This notebook provides a deep dive into implementing observability layers for multi-agent LLM systems, specifically focusing on the **Cross-Indiscretion Matrix** and **Agentic RAG State Machines**.

These implementations translate theoretical concepts into functional python loops and tracking mechanisms.

## 1. Multi-Agent Pipeline Setup
We start by simulating a standard agentic framework where specialized models handle discrete tasks: Routing, Retrieving, Synthesizing, and Evaluating.

In [ ]:
import random
import time
from typing import Dict, List, Any

class Agent:
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role

    def process(self, payload: dict) -> dict:
        # Mock processing logic
        print(f"[{self.name} - {self.role}] processing payload...")
        time.sleep(0.5) # Simulate latency
        return {**payload, 'processed_by': self.name}

# Initialize Pipeline Agents
router = Agent('Agent-R', 'Router')
retriever = Agent('Agent-D', 'Retriever')
synthesizer = Agent('Agent-S', 'Synthesizer')
evaluator = Agent('Agent-E', 'Evaluator')

## 2. The Cross-Indiscretion Matrix
The Cross-Indiscretion Matrix acts as an audit layer. It sits outside the pipeline and logs handoffs. If one agent alters the context unexpectedly or introduces hallucinations, the matrix calculates a 'discrepancy score' and flags it.

In [ ]:
class CrossIndiscretionMatrix:
    def __init__(self):
        self.logs = []
        self.threshold = 0.7 # Severity threshold for raising alerts
    
    def log_interaction(self, source: str, target: str, payload: dict, discrepancy_score: float, context: str):
        """
        Logs the edge interaction between two agents in the pipeline.
        """
        interaction = {
            'timestamp': time.time(),
            'source': source,
            'target': target,
            'score': discrepancy_score,
            'context': context,
            'flagged': discrepancy_score > self.threshold
        }
        self.logs.append(interaction)
        
        if interaction['flagged']:
            print(f"🚨 INDISCRETION FLAG: {source} -> {target} | Score: {discrepancy_score} | Context: {context}")
        else:
            print(f"✅ Safe Handoff: {source} -> {target} | Score: {discrepancy_score}")

# Initialize and simulate pipeline errors
matrix = CrossIndiscretionMatrix()

print("--- Simulating Pipeline Handoffs ---")
matrix.log_interaction('Router', 'Retriever', {'query': 'Find XYZ specs'}, 0.1, 'Standard handoff')
matrix.log_interaction('Retriever', 'Synthesizer', {'docs': [1,2,3]}, 0.2, 'Context delivered')

# Simulate Context Drift & Hallucination
matrix.log_interaction('Synthesizer', 'Evaluator', {'ans': 'XYZ has 500V (fake)'}, 0.85, 'Hallucination Detected')

## 3. Agentic RAG State Machine
Here is the implementation of the evaluative RAG `while` loop. Instead of failing when bad context is retrieved, the system uses a 'Grader' to conditionally trigger query rewriting strategies (like HyDE or Step-Back Prompting).

In [ ]:
def mock_vector_search(query: str, attempt: int):
    # We simulate a scenario where the database only returns the right context
    # after the query has been heavily rewritten (attempt 3)
    if attempt >= 3:
        return f"[Relevant Doc 404] Specs found for expanded query: '{query}'"
    return "[Doc 101] Unrelated marketing material."

def llm_grade_context(context: str):
    # Mock LLM Grader logic
    return "Specs found" in context

def llm_rewrite_query(query: str, strategy: str):
    # Mock rewriting strategies
    if strategy == "step_back":
        return f"{query} -> Abstracted to broader product category"
    elif strategy == "hyde":
        return f"{query} -> Hypothetical answer embedding approach"
    return query

def agentic_rag_loop(initial_query: str, max_retries: int = 4):
    current_query = initial_query
    strategies = ["step_back", "hyde", "multi_hop"]
    
    for attempt in range(1, max_retries + 1):
        print(f"\n[Loop {attempt}] Executing Vector Search...")
        print(f"Query State: '{current_query}'")
        
        # Step 1: Retrieve
        context = mock_vector_search(current_query, attempt)
        print(f"Retrieved: {context}")
        
        # Step 2: Grade
        is_relevant = llm_grade_context(context)
        
        # Step 3: Conditional Routing
        if is_relevant:
            print("✅ Grader Status: Pass. Routing to Synthesizer.")
            return f"Final Output Generated based on: {context}"
        else:
            print("❌ Grader Status: Fail. Context irrelevant.")
            if attempt < max_retries:
                strat = strategies[(attempt-1) % len(strategies)]
                print(f"🔄 Routing to Rewriter (Strategy: {strat})")
                current_query = llm_rewrite_query(current_query, strat)
            
    return "Loop Exited: Reached max_retries without finding relevant context. Failing safely."

# Run the loop
print("=== STARTING AGENTIC RAG PIPELINE ===")
final_answer = agentic_rag_loop("What is the sub-system latency of the XYZ architecture?")
print(f"\n=== PIPELINE FINISHED ===\nResult: {final_answer}")

## 4. Visualizing the Logic Graph
Using NetworkX to build a topological view of the RAG State Machine nodes and conditional edges.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Define the State Machine Graph
G = nx.DiGraph()

# Add directed edges based on the RAG loop logic
edges = [
    ('User Input', 'Router Agent'),
    ('Router Agent', 'Retriever'),
    ('Retriever', 'Evaluator (Grader)'),
    ('Evaluator (Grader)', 'Synthesizer (Output)'),  # Path if Context == Good
    ('Evaluator (Grader)', 'Query Rewriter'),        # Path if Context == Bad
    ('Query Rewriter', 'Retriever'),                 # The Loop Back
    ('Synthesizer (Output)', 'Evaluator (Grader)')   # Final hallucination check loop
]
G.add_edges_from(edges)

# Plotting the Graph
plt.figure(figsize=(12, 7))
pos = nx.spring_layout(G, seed=42)  # Layout algorithm

# Draw nodes and edges
nx.draw_networkx_nodes(G, pos, node_color='#87CEEB', node_size=3500, edgecolors='black')
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, width=2)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')

plt.title("Agentic RAG State Machine (Conditional Edges)", fontsize=16)
plt.axis('off') # Hide the axes
plt.tight_layout()
plt.show()